In [3]:
import pandas as pd
import numpy as np

# -------------------------------
# STEP 1: LOAD DATA
# -------------------------------
df = pd.read_csv("prediction_log.csv")

# -------------------------------
# STEP 2: FIX PREDICTION COLUMN
# -------------------------------
df["prediction"] = df["prediction"].map({
    "Approved": 1,
    "Rejected": 0
})

# remove any missing values
df = df.dropna(subset=["prediction"])

# -------------------------------
# STEP 3: CLEAN TIMESTAMP
# -------------------------------
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])

# -------------------------------
# STEP 4: SIMULATE MULTIPLE DAYS (VERY IMPORTANT)
# -------------------------------
df["timestamp"] = df["timestamp"] - pd.to_timedelta(
    np.random.randint(0, 10, size=len(df)), unit="D"
)

df["date"] = df["timestamp"].dt.date

# -------------------------------
# STEP 5: SIMULATE ACTUAL VALUES
# -------------------------------
np.random.seed(42)
df["actual"] = np.random.choice([0, 1], size=len(df))

# -------------------------------
# STEP 6: CREATE ACCURACY METRIC
# -------------------------------
df["correct"] = (df["prediction"] == df["actual"]).astype(int)

# -------------------------------
# STEP 7: FIX COLUMN NAMES
# -------------------------------
df.rename(columns={
    "applicant_income": "Applicant_Income",
    "loan_amount": "Loan_Amount"
}, inplace=True)

# -------------------------------
# STEP 8: AGGREGATE METRICS
# -------------------------------
daily_metrics = df.groupby("date").agg({
    "correct": "mean",
    "prediction": "mean",
    "Applicant_Income": "mean",
    "Loan_Amount": "mean"
}).reset_index()

# -------------------------------
# STEP 9: RENAME FOR DASHBOARD
# -------------------------------
daily_metrics.rename(columns={
    "correct": "accuracy",
    "prediction": "approval_rate",
    "Applicant_Income": "avg_income",
    "Loan_Amount": "avg_loan_amount"
}, inplace=True)

# -------------------------------
# STEP 10: SORT DATA
# -------------------------------
daily_metrics.sort_values("date", inplace=True)

# -------------------------------
# STEP 11: OUTPUT
# -------------------------------
print(daily_metrics.head())

# -------------------------------
# STEP 12: SAVE FILE FOR TABLEAU
# -------------------------------
daily_metrics.to_csv("monitoring_dashboard.csv", index=False)

print("✅ Monitoring dashboard file created!")

         date  accuracy  approval_rate  avg_income  avg_loan_amount
0  2026-03-24       0.5            0.5    875000.0        5250000.0
1  2026-03-25       1.0            0.0    100000.0        4000000.0
2  2026-03-26       1.0            0.0   1000000.0        7000000.0
3  2026-03-27       0.5            0.5    525000.0        2500000.0
4  2026-03-28       1.0            0.0   1000000.0        7000000.0
✅ Monitoring dashboard file created!
